In [1]:
# ==========================================================================
# GitHub表示用（一番最初に実行してください）
# ==========================================================================
import matplotlib.pyplot as plt
import pandas as pd
import warnings

# グラフのダークモード対策（背景に薄いグレーのグリッドを敷く）
plt.style.use('ggplot') 

# データフレーム（表）の表示行数・列数を最大10に制限してスッキリさせる
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', 10)

# 不要な警告（Warning）を非表示にする
warnings.filterwarnings('ignore')

In [ ]:
## 有名なタイタニック号の乗客の生存データを使ってデータ分析の勉強を実施
# 分かったことや間違えたことなどの記録用

In [2]:
import seaborn as sns
import pandas as pd
df = sns.load_dataset("titanic")
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [3]:
#当初、文字列を数字に置換するためのリストを作って、相関係数を見ようと思ったが、調べてみるとこの方法では結果に悪さをすることが分かった。
mapping_dict = {'sex' : {'male' : 0, 'female' : 1},
                          'embarked':{'S' :0, 'C' : 1, 'Q' : 2},
                          'who':{'man' : 0, 'woman' : 1, 'child' : 2},
                          'adult_male' : {True : 1,False : 0},
                          'alone' : {True : 1, False : 0},
                          'deck' : {'A' : 0, 'B' : 1, 'C' : 2, 'D' : 3, 'E' : 4, 'F' : 5, 'G' : 6}}


In [4]:
#相関係数を正しく見るには、文字列をget.dummiesで変換する
analysis_df = pd.get_dummies(df, columns=['sex','who','embarked','adult_male','deck','embark_town'], drop_first=False)
#分解した状態のまま、相関係数を計算する
# survivedの列と、他の列との相関だけを抜き出して見やすくする
target_corr = analysis_df.corr(numeric_only=True)['survived'].sort_values(ascending=False)
target_corr.head()

survived            1.000000
adult_male_False    0.557080
sex_female          0.543351
who_woman           0.506562
fare                0.257307
Name: survived, dtype: float64

In [5]:
#多重共線性をseabornのヒートマップで確認したが、3つ以上の変数が複雑に絡み合っている場合を見落とすという弱点があるため、VIFを行った方が良いということが分かった

In [8]:
#VIF（Variance Inflation Factor、分散拡大係数）は、重回帰分析において説明変数間の「多重共線性（マルチコ）」の深刻さを評価する指標
#値が大きいほど他変数との相関が強く、一般的に10（または5）以上で多重共線性が疑われ、該当変数の削除や統合が検討される

from statsmodels.stats.outliers_influence import variance_inflation_factor

#---意味が完全に重複している列と目的変数を削除 ---
duplicated_columns = ['adult_male', 'embark_town', 'class', 'alive', 'survived']
analysis_df = df.drop(columns=duplicated_columns).copy()

# ---先にcategory型を普通の文字列に変換する---
for col in analysis_df.columns:
    if analysis_df[col].dtype.name == 'category':
        analysis_df[col] = analysis_df[col].astype(str)

# ---欠損値を処理する ---
# 年齢は中央値で埋める
analysis_df['age'] = analysis_df['age'].fillna(analysis_df['age'].median())
# デッキや港などの文字データの空っぽを「Unknown（不明）」という文字で埋める
analysis_df = analysis_df.fillna("Unknown")

# 文字列を 0 と 1 に直し、全体を float 型に変換する
# drop_first=True は、ダミー変数の中から、あえて最初の1本を消すことで、データの重複をなくす設定
# 重回帰分析をするときは、 数式がバグるのを防ぐため、必ず True にする
# ランダムフォレストや決定木、相関係数を見るときは、人間が見て分かりやすいように False で全種類残してもOK
analysis_df_encoded = pd.get_dummies(analysis_df, drop_first=True).astype(float)

# 定数項（基準の列）を足す
# すべての行に 1.0 を入れるのは、AIに「全員共通の切片・基本点」を正しく計算させるための、数式上のテクニック
analysis_df_with_const = analysis_df_encoded.copy()
analysis_df_with_const['const'] = 1.0

# 全カラムのVIFを計算
vif_data = pd.DataFrame()
vif_data["特徴量"] = analysis_df_with_const.columns
vif_data["VIF値"] = [
    variance_inflation_factor(analysis_df_with_const.values, i)
    for i in range(len(analysis_df_with_const.columns))
]
# VIF値が大きい順に並び替えて、人間が見やすくする
vif_data = vif_data.sort_values(by="VIF値", ascending=False).reset_index(drop=True)

# 結果を表示
vif_data.head()

,特徴量,VIF値
0,const,100.102361
1,deck_nan,12.597181
2,sex_male,10.269211
3,who_man,8.728388
4,who_woman,6.547739


In [ ]:
# まずは、重回帰分析を実施する
# IterativeImputerは、欠損値を推定して埋めるためのAIで、年齢のような数値データの欠損を埋めるのに適しているため、今回はこれを使う
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import r2_score

# 必要なカラムだけ選んでSurvivedのカラムが欠損している行は消すという処理
analysis_df = df[['pclass', 'fare', 'alone', 'sex', 'age', 'survived']].dropna(subset=['survived']).copy()

# データを目的変数と説明変数に分けて、学習用とテスト用に分割する
X = analysis_df.drop(columns=['survived'])
y = analysis_df['survived']
# 習用とテスト用に分割する際、ランダムに分けるための乱数の種（random_state）を指定しておくと、毎回同じ分け方になるので、結果の再現性が高まる
# test_size=0.2 は、全体の20%をテスト用にするという意味、20-30%くらいが一般的な割合
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 分けたあとに、文字列（sex）を 0 と 1 の数字に変換する
# drop_first=True は、ダミー変数の中から、あえて最初の1本を消すことで、データの重複をなくす設定
X_train = pd.get_dummies(X_train, columns=['sex'], drop_first=True)
X_test = pd.get_dummies(X_test, columns=['sex'], drop_first=True)

# AI（IterativeImputer）で、1人ずつ年齢を推定して埋める
# グループを示す文字列のような場合にはこの方法を使えないので注意
imputer = IterativeImputer(max_iter=10, random_state=42)
X_train_filled = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test_filled = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

# スケール（桁数）の補正
# fit_transform は、学習用データを使ってスケーラーを学習し、その学習した内容をもとに学習用データを変換するという意味
# transform は、学習用データを使って学習した内容をもとにテスト用データを変換するという意味
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_filled)
X_test_scaled = scaler.transform(X_test_filled)

# 重回帰分析の学習
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# スコアの計算をする前に、テスト用データを使って予測をしておく
# 予測をする前に、テスト用データを使って予測をしておくのは、スコアの計算をするために必要なステップ
# R2スコアは、予測がどれだけ正解に近いかを示す指標で、1に近いほど予測が正解に近いことを意味する
y_pred = model.predict(X_test_scaled)
print(f"修正後のR2スコア: {r2_score(y_test, y_pred):.3f}")

修正後のR2スコア: 0.440


In [ ]:
#同じく重回帰分析だが、文字列の欠損値もAIで予測補完してみるため、HistGradientBoostingRegressorを使ったIterativeImputerを作成してみる
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# 重複列とターゲットを削除
# .copy()でデータをコピーして、元のデータを変更しないようにする
duplicated_columns = ['adult_male', 'embark_town', 'class', 'alive', 'survived']
X = df.drop(columns=duplicated_columns).copy()
y = df['survived']

# category型を普通の文字列（str）に変換しておく
for col in X.columns:
    if X[col].dtype.name == 'category':
        X[col] = X[col].astype(str)

# データを「練習用」と「テスト用」に分ける
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 文字列の列をダミー変数に分解して、すべてを float 型に変換する
# drop_first=False にすることで、すべてのダミー変数を残し、欠損値の行はすべて0になるようにする
X_train_encoded = pd.get_dummies(X_train, drop_first=False).astype(float)
X_test_encoded = pd.get_dummies(X_test, drop_first=False).astype(float)

# 練習用とテスト用で列のズレが起きないようにピタッと揃える
# align() は、2つのDataFrameの列を揃えるための便利な関数
# #join='left' と axis=1 を指定することで、X_train_encodedの列を基準にして、X_test_encodedの列を揃えます
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0.0)

# pd.get_dummiesを使って文字を数字に変換すると、NaNが、勝手に『0』に化けてしまう
# 数字のデータは pd.get_dummies を使っても、NaNは NaN のまま、きれいに残る
# 勝手に0になっけど、元々はNaNだから、ちゃんと（np.nan）』に戻すという作業をしている
# 元データ（X_train / X_test）でNaNだった行を見つけ出し、
# 分解後の対応する列を一括で np.nan （空っぽ）に直接上書きする
for col in X.columns:
    if X[col].dtype == 'object':
        # 元の列（例: 'deck'）で欠損値だった行のインデックス（番号）を取得
        train_nan_idx = X_train[X_train[col].isna()].index
        test_nan_idx = X_test[X_test[col].isna()].index

        # 分解後のデータから、その変数に関連する列（例: 'deck_A', 'deck_B' など）を安全に見つける
        # 文字列として安全に比較するため、一度列名をstr型にキャストします
        target_cols = [c for c in X_train_encoded.columns if str(c).startswith(f"{col}_")]

        # 元々空っぽだった行の、対象の列をすべて np.nan に上書き（これでAIが欠損値として認識できます！）
        if target_cols:
            X_train_encoded.loc[train_nan_idx, target_cols] = np.nan
            X_test_encoded.loc[test_nan_idx, target_cols] = np.nan

# HistGradientBoostingを使ったIterativeImputerを作成
# HistGradientBoostingは、決定木をベースにした強力な回帰モデルで、欠損値の推定にも優れた性能を発揮します

advanced_imputer = IterativeImputer(
    estimator=HistGradientBoostingRegressor(
        max_depth=5,         # 深読みしすぎないように制限、過学習を防止、5くらいがちょうどいい
        learning_rate=0.05,  # デフォルト（0.1）の半分。慎重に学習、0.01とかにするとさらに慎重になるけど、時間もかかるので注意
        random_state=42
    ),
    max_iter=20,             # 慎重にやる分、繰り返し回数を10回から20回に増やす！増やすほど精度が上がる可能性があるけど、時間もかかるので注意
    random_state=42
)

#advanced_imputer = IterativeImputer(
    #estimator=RandomForestRegressor(n_estimators=100, random_state=42), # ➔ 別の大物モデルに変更！
    #max_iter=10,
    #random_state=42
#)

# --- データを分けた直後のゾーン ---

# X_trainだけで、年齢の平均値を計算する
# テストデータの情報を使わないように、必ず練習用データだけで計算することが重要
age_mean = X_train['age'].mean()
# 練習用データに、新しいヒント「age_diff（平均との歳の差）」を足す
# ※ .copy()の後に安全に列を追加するために .loc[:, ...] を使うのが適切
X_train.loc[:, 'age_diff'] = X_train['age'] - age_mean
# テストデータにも「全く同じ練習用の平均値（age_mean）」を使って列を足す！
# X_test.mean() を使うのは絶対ダメ！テストデータの情報を使ってしまうことになるので、必ず練習用の平均値（age_mean）を使うこと！
X_test.loc[:, 'age_diff'] = X_test['age'] - age_mean

# AIを実行して、年齢（数字）もデッキ（文字のダミー）も一斉に予測補完！
X_train_filled = pd.DataFrame(advanced_imputer.fit_transform(X_train_encoded), columns=X_train_encoded.columns)
X_test_filled = pd.DataFrame(advanced_imputer.transform(X_test_encoded), columns=X_test_encoded.columns)

# データの桁数を揃える（スケール補正）
scaler = StandardScaler()
X_train_scaled_lr = scaler.fit_transform(X_train_filled)
X_test_scaled_lr = scaler.transform(X_test_filled)

# 重回帰分析のAIに学習させる
model = LinearRegression()
model.fit(X_train_scaled_lr, y_train)

# スコアとランキングの発表
y_pred = model.predict(X_test_scaled_lr)
print(f"文字列もAIで予測補完したモデルのR2スコア: {r2_score(y_test, y_pred):.3f}")

# 最後にランキングを表示
# 列名が万が一タプル形式になっていても綺麗に読めるように文字列に直します
coefficients = pd.DataFrame({
    'ヒント名': [str(c) for c in X_train_filled.columns],
    '重要度（係数）': model.coef_
}).sort_values(by='重要度（係数）', key=abs #絶対値でソートするための設定
               , ascending=False)

print("\n=== AIが重要だと判断したヒントランキング（上位5つ） ===")
print(coefficients.head(5).to_string(index=False))

文字列もAIで予測補完したモデルのR2スコア: 0.434

=== AIが重要だと判断したヒントランキング（上位5つ） ===
     ヒント名   重要度（係数）
  who_man -0.138016
who_woman  0.109476
    sibsp -0.085920
   pclass -0.079791
who_child  0.058903


In [ ]:
# まとめて色々試す方法）
models = {
    '重回帰分析': LinearRegression(),
    'ランダムフォレスト': RandomForestRegressor(random_state=42),
    'ヒストグラディエントブースティング': HistGradientBoostingRegressor(random_state=42)
}

# 結果を記録する箱
results = {}

# 自動で学習・予測・採点をループ！
for name, model in models.items():
    # 重回帰分析だけは、桁数を揃えた「_scaled_lr」のデータを渡す
    if name == '重回帰分析':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    # 決定木系は、そのままの「_filled」のデータを渡す
    else:
        model.fit(X_train_filled, y_train)
        y_pred = model.predict(X_test_filled)
    
    # 採点して記録
    results[name] = r2_score(y_test, y_pred)

print("\n=== AIロボット別 R2スコア選手権 ===")
for name, score in results.items():
    print(f"{name} のスコア: {score:.3f}")


=== AIロボット別 R2スコア選手権 ===
重回帰分析 のスコア: 0.434
ランダムフォレスト のスコア: 0.435
ヒストグラディエントブースティング のスコア: 0.450


In [12]:
import joblib
# AIの保存
joblib.dump(advanced_imputer, 'my_taitanic_imputer.pkl')
joblib.dump(scaler, 'my_taitanic_scaler.pkl')
joblib.dump(model, 'my_titanic_model.pkl')
print("AIをファイルに保存しました！")

AIをファイルに保存しました！


In [17]:
# 実際に使うための方法を整理
# 保存しておいたAIの頭脳をロード（読み込み）する
loaded_imputer = joblib.load('my_taitanic_imputer.pkl')
loaded_scaler = joblib.load('my_taitanic_scaler.pkl')
loaded_model = joblib.load('my_titanic_model.pkl')

# 新しい乗客のデータを仮で作成
new_data = pd.DataFrame([{
    'pclass': 1, 'sex': 'female', 'age': None,
    'sibsp': 0, 'parch': 0, 'fare': 150.0, 'embarked': 'S', 'deck': 'C', 'alone': True
}])

# 前処理（0と1の分解）を、当時と同じルールで行う
new_data_encoded = pd.get_dummies(new_data).astype(float)
# 当時の練習用データの列並び（X_train_encoded.columns）にピタッと強制的に揃える
new_data_encoded = new_data_encoded.reindex(columns=X_train_encoded.columns, fill_value=0.0)

# 保存されたAIを使って、穴埋め ➔ 桁揃え ➔ 生存予測！
new_data_filled = loaded_imputer.transform(new_data_encoded) # 年齢をAIが自動で推測！
new_data_scaled = loaded_scaler.transform(new_data_filled)
prediction = loaded_model.predict(new_data_scaled)

print(f"この新しい乗客の生存予測確率: {prediction[0] * 100:.1f}%")

この新しい乗客の生存予測確率: 77.4%


c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but HistGradientBoostingRegressor was fitted with feature names
  warnings.warn(


In [ ]:
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# データの読み込みと準備
duplicated_columns = ['adult_male', 'embark_town', 'class', 'alive', 'survived']
X = df.drop(columns=duplicated_columns).copy()
y = df['survived']

#ここを修正！先にcategory型の縛りを解く
for col in X.columns:
    if X[col].dtype.name == 'category':
        X[col] = X[col].astype(str)

# 年齢の欠損値を中央値で埋める
X['age'] = X['age'].fillna(X['age'].median())
# 残りの文字列の空っぽを「Unknown」で埋める（もうエラーになりません！）
X = X.fillna("Unknown")

# 決定木・ランダムフォレストを動かすために、文字列を 0 と 1（ダミー変数）に変形
X_encoded = pd.get_dummies(X, drop_first=True).astype(float)

# 2. 練習用とテスト用に分割
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# 条件分岐の質問を最大3回までにする制限（max_depth=3）
tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_model.fit(X_train, y_train)

y_tree_pred = tree_model.predict(X_test)
print(f"決定木のモデル精度（正解率）      : {accuracy_score(y_test, y_tree_pred) * 100:.1f}%")

# ランダムフォレストの実行
# 100本の木を作ってチームで多数決（max_depth=5）
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)

y_rf_pred = rf_model.predict(X_test)
print(f"ランダムフォレストの精度（正解率）: {accuracy_score(y_test, y_rf_pred) * 100:.1f}%")

🌲 決定木のモデル精度（正解率）      : 81.0%
🔥 ランダムフォレストの精度（正解率）: 80.4%


In [18]:
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# データの準備と前処理（エラー回避済）
duplicated_columns = ['adult_male', 'embark_town', 'class', 'alive', 'survived']
X = df.drop(columns=duplicated_columns).copy()
y = df['survived']

for col in X.columns:
    if X[col].dtype.name == 'category':
        X[col] = X[col].astype(str)

X['age'] = X['age'].fillna(X['age'].median())
X = X.fillna("Unknown")
X_encoded = pd.get_dummies(X, drop_first=True).astype(float)

X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# 両社のモデルを訓練
tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_model.fit(X_train, y_train)

rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)

# =========================================================================
# 比較①：スコア（正解率）を一覧表で比べる
# =========================================================================
tree_acc = accuracy_score(y_test, tree_model.predict(X_test)) * 100
rf_acc = accuracy_score(y_test, rf_model.predict(X_test)) * 100

score_compare = pd.DataFrame({
    'モデル名': ['決定木（ソロ）', 'ランダムフォレスト（チーム）'],
    'テストデータの正解率': [f"{tree_acc:.1f}%", f"{rf_acc:.1f}%"]
})

print("=== 比較：正解率（Accuracy）の比較 ===")
print(score_compare.to_string(index=False))
print("\n" + "="*50 + "\n")

# =========================================================================
# 比較②：AIが重要視した「特徴量」の比較
# =========================================================================
# 各モデルが計算した「重要度（feature_importances_）」を抜き出します
importances = pd.DataFrame({
    '特徴量': X_train.columns,
    '決定木の重要度': tree_model.feature_importances_,
    'ランダムフォレストの重要度': rf_model.feature_importances_
})

print("=== 比較：両社が重要だと見抜いた特徴量（上位5つ） ===")
print("【決定木が選んだトップ5】")
print(importances.sort_values(by='決定木の重要度', ascending=False).head(5)[['特徴量', '決定木の重要度']].to_string(index=False))
print("\n【ランダムフォレストが選んだトップ5】")
print(importances.sort_values(by='ランダムフォレストの重要度', ascending=False).head(5)[['特徴量', 'ランダムフォレストの重要度']].to_string(index=False))
print("\n" + "="*50 + "\n")

# =========================================================================
# 比較：特定の乗客（例：テストの先頭5人）への予測結果を直接並べて比べる
# =========================================================================
test_passengers = X_test.head(5)
true_answers = y_test.head(5).values

# 生存確率（1になる確率）をそれぞれ計算
tree_probs = tree_model.predict_proba(test_passengers)[:, 1] * 100
rf_probs = rf_model.predict_proba(test_passengers)[:, 1] * 100

passenger_compare = pd.DataFrame({
    '実際の正解': ['生存' if a == 1 else '死亡' for a in true_answers],
    '決定木の生存予測確率': [f"{p:.1f}%" for p in tree_probs],
    'ランダムフォレストの生存予測確率': [f"{p:.1f}%" for p in rf_probs]
})

print("===比較：実際の乗客5人に対する生存確率の予測比較 ===")
print(passenger_compare)

=== 比較：正解率（Accuracy）の比較 ===
          モデル名 テストデータの正解率
       決定木（ソロ）      81.0%
ランダムフォレスト（チーム）      80.4%


=== 比較：両社が重要だと見抜いた特徴量（上位5つ） ===
【決定木が選んだトップ5】
         特徴量  決定木の重要度
     who_man 0.617919
      pclass 0.215694
        fare 0.114152
deck_Unknown 0.049807
      deck_C 0.002428

【ランダムフォレストが選んだトップ5】
      特徴量  ランダムフォレストの重要度
 sex_male       0.214668
  who_man       0.202921
who_woman       0.148459
   pclass       0.095194
     fare       0.083011


===比較：実際の乗客5人に対する生存確率の予測比較 ===
  実際の正解 決定木の生存予測確率 ランダムフォレストの生存予測確率
0    生存      10.4%            14.5%
1    死亡      10.4%            11.8%
2    死亡      10.4%            12.2%
3    生存      98.2%            79.7%
4    生存      59.6%            54.5%


In [20]:
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# 1. 過去のデータ（タイタニック号の記録）を読み込んで学習の準備
df = sns.load_dataset("titanic")
duplicated_columns = ['adult_male', 'embark_town', 'class', 'alive', 'survived']
X = df.drop(columns=duplicated_columns).copy()
y = df['survived']

# category型の縛りを解いて、欠損値を穴埋めする（いつもの安全な前処理）
for col in X.columns:
    if X[col].dtype.name == 'category':
        X[col] = X[col].astype(str)
X['age'] = X['age'].fillna(X['age'].median())
X = X.fillna("Unknown")

# 文字列を 0 と 1 に分解する
X_encoded = pd.get_dummies(X, drop_first=True).astype(float)

# 2. ランダムフォレスト（分類AI）に過去のデータをすべて覚えさせる
# ※今回は未来の予測が目的なので、贅沢にデータ全量を使ってAIを最強に鍛えます！
final_rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
final_rf_model.fit(X_encoded, y)
print("過去のデータから、生存パターンの学習が完了しました！\n" + "="*60)

# =========================================================================
# ここから「未来の新しい乗客（正解のわからないデータ）」を作る
# =========================================================================
# 例として、タイタニック号に新しく乗船した想定の「3人の乗客データ」を自作します
new_passengers = pd.DataFrame([
    {
        'pclass': 1, 'sex': 'female', 'age': 25.0, 'sibsp': 0, 'parch': 0,
        'fare': 150.0, 'embarked': 'S', 'deck': 'B', 'alone': True
    }, # ① お金持ちの25歳女性（1等席・Bデッキ）
    {
        'pclass': 3, 'sex': 'male', 'age': 30.0, 'sibsp': 0, 'parch': 0,
        'fare': 7.5, 'embarked': 'S', 'deck': 'Unknown', 'alone': True
    }, # ② 格安チケットの30歳男性（3等席・お部屋不明）
    {
        'pclass': 2, 'sex': 'female', 'age': 35.0, 'sibsp': 1, 'parch': 2,
        'fare': 26.0, 'embarked': 'C', 'deck': 'Unknown', 'alone': False
    }  # ③ 家族連れの35歳女性（2等席・フランスから乗船）
], index=['乗客A（富裕層女性）', '乗客B（一般男性）', '乗客C（家族連れ女性）'])

# =========================================================================
# 新しいデータも、当時と同じ「0と1の形」に形を整える（超重要！）
# =========================================================================
new_passengers_encoded = pd.get_dummies(new_passengers).astype(float)

# 過去のデータで作った列の並び（X_encoded.columns）と「完全に同じ」になるように並び替える
# 新しいデータに存在しない列（例：deck_Cなど）には、自動的に 0.0 が入ります
new_passengers_encoded = new_passengers_encoded.reindex(columns=X_encoded.columns, fill_value=0.0)

# =========================================================================
# 未来の乗客を「分類予測」する！
# =========================================================================
# 【予測①】生存(1)か死亡(0)かの、2択のファイナルアンサーを出す命令
predictions = final_rf_model.predict(new_passengers_encoded)

# 【予測②】裏側で、AIがどれくらい確信しているかの「確率」を出す命令（1になる確率を％にする）
probabilities = final_rf_model.predict_proba(new_passengers_encoded)[:, 1] * 100

# 結果を綺麗に見せるために表にまとめる
result_df = pd.DataFrame({
    'AIの予測': ['生存' if p == 1 else '死亡' for p in predictions],
    'AIが予測した生存確率': [f"{prob:.1f}%" for prob in probabilities]
}, index=new_passengers.index)

print("\n=== 🔮 新しい乗客3人に対する、AIの未来予測結果 ===")
result_df

過去のデータから、生存パターンの学習が完了しました！

=== 🔮 新しい乗客3人に対する、AIの未来予測結果 ===


,AIの予測,AIが予測した生存確率
乗客A（富裕層女性）,生存,85.4%
乗客B（一般男性）,死亡,39.7%
乗客C（家族連れ女性）,生存,73.5%
